# Bayes-Factor Validation Storyboard

Goal: plan the second validation stage after fitting the most complex model.

We want two evidence stories across cumulative cell number:

1. **Eta heterogeneity**: does the model recover whether decision propensity varies across cells? Compare models with `sigma_eta = 0` against models with `sigma_eta > 0` using SMC log evidence.
2. **History effects**: does the model recover whether `beta_x` and `beta_y` are zero? Use Savage-Dickey ratios inside the history-dependent model when prior draws are available.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def find_section_root(start: Path) -> Path:
    start = start.resolve()
    for path in [start, *start.parents]:
        if (path / "section_3" / "src" / "inference.py").exists():
            return path / "section_3"
        if path.name == "section_3" and (path / "src" / "inference.py").exists():
            return path
    raise FileNotFoundError("Could not locate section_3 root.")


section_root = find_section_root(Path.cwd())
figure_dir = section_root / "figures" / "bf_validation_storyboard"
figure_dir.mkdir(parents=True, exist_ok=True)

PARAM_LABELS = {
    "sigma_eta": r"$\sigma_\eta$",
    "beta_x": r"$\beta_x$",
    "beta_y": r"$\beta_y$",
}

section_root

## Coding Story

A clean implementation can be split into three steps.

### Step A: synthetic scenarios

Use four scenarios so every comparison has a true positive and a true negative case:

| scenario | `sigma_eta` | `beta_x`, `beta_y` | true story |
|---|---:|---:|---|
| `No1_eta_beta` | nonzero | nonzero | heterogeneous and history-dependent |
| `No2_eta_only` | nonzero | zero | heterogeneous, no history effect |
| `No3_beta_only` | zero | nonzero | homogeneous eta, history-dependent |
| `No4_null` | zero | zero | homogeneous eta, no history effect |

### Step B: cumulative sample sizes

For each scenario, generate one maximum-size synthetic population, then fit cumulative slices such as:

`10, 20, 30, 50, 100, 200, 400, 500, 1000`.

### Step C: model fits and evidence

Fit the four section 3 decision models for each slice:

- `homogeneous_history_independent`: `sigma_eta = 0`, `beta = 0`
- `homogeneous_history_dependent`: `sigma_eta = 0`, beta free
- `heterogeneous_history_independent`: `sigma_eta > 0`, `beta = 0`
- `heterogeneous_history_dependent`: `sigma_eta > 0`, beta free

Then save one summary table with log evidence and derived Bayes factors.

In [ ]:
SCENARIOS = [
    {
        "scenario": "No1_eta_beta",
        "label": "eta heterogeneity + history effects",
        "sigma_eta": 0.75,
        "beta_x": 0.8,
        "beta_y": -0.8,
        "true_eta": "heterogeneous",
        "true_beta": "nonzero",
    },
    {
        "scenario": "No2_eta_only",
        "label": "eta heterogeneity only",
        "sigma_eta": 0.75,
        "beta_x": 0.0,
        "beta_y": 0.0,
        "true_eta": "heterogeneous",
        "true_beta": "zero",
    },
    {
        "scenario": "No3_beta_only",
        "label": "history effects only",
        "sigma_eta": 0.0,
        "beta_x": 0.8,
        "beta_y": -0.8,
        "true_eta": "homogeneous",
        "true_beta": "nonzero",
    },
    {
        "scenario": "No4_null",
        "label": "homogeneous, no history effects",
        "sigma_eta": 0.0,
        "beta_x": 0.0,
        "beta_y": 0.0,
        "true_eta": "homogeneous",
        "true_beta": "zero",
    },
]

MODELS = [
    "homogeneous_history_independent",
    "homogeneous_history_dependent",
    "heterogeneous_history_independent",
    "heterogeneous_history_dependent",
]

pd.DataFrame(SCENARIOS)

## Eta-Heterogeneity Bayes Factor

For eta heterogeneity, use SMC log evidence because `sigma_eta = 0` is a boundary/null model and Savage-Dickey is not the cleanest tool here.

A simple evidence contrast is:

`log_BF_eta = max(logml of heterogeneous models) - max(logml of homogeneous models)`

Positive values support eta heterogeneity. In log10 units:

`log10_BF_eta = log_BF_eta / log(10)`

In [ ]:
def compute_eta_bf(logml_summary: pd.DataFrame) -> pd.DataFrame:
    rows = []

    group_cols = ["scenario", "replicate", "n_cell"]
    available_group_cols = [col for col in group_cols if col in logml_summary.columns]

    for keys, group in logml_summary.groupby(available_group_cols):
        if not isinstance(keys, tuple):
            keys = (keys,)
        key_data = dict(zip(available_group_cols, keys))

        hetero = group[group["model"].str.startswith("heterogeneous")]
        homo = group[group["model"].str.startswith("homogeneous")]
        if hetero.empty or homo.empty:
            continue

        hetero_logml = float(hetero["logml"].max())
        homo_logml = float(homo["logml"].max())
        log_bf = hetero_logml - homo_logml

        rows.append(
            {
                **key_data,
                "hetero_logml_best": hetero_logml,
                "homo_logml_best": homo_logml,
                "log_bf_eta_hetero_vs_homo": log_bf,
                "log10_bf_eta_hetero_vs_homo": log_bf / np.log(10.0),
            }
        )

    return pd.DataFrame(rows)


# Later, after creating section_3/results/part_2_bf_trajectory/logml_summary.csv:
# logml_summary = pd.read_csv(section_root / "results" / "part_2_bf_trajectory" / "logml_summary.csv")
# eta_bf = compute_eta_bf(logml_summary)
# display(eta_bf.head())

## Beta Savage-Dickey Trajectory

For `beta_x = 0` and `beta_y = 0`, use Savage-Dickey inside the full history-dependent model. This is already available in the current part 1 outputs, so we can test the plotting shape immediately.

In [ ]:
latest_path = section_root / "results" / "part_1_complex_latest.txt"
run_rel = latest_path.read_text().strip()
current_run_dir = section_root / "results" / run_rel

rows = []
for n_dir in sorted([p for p in current_run_dir.iterdir() if p.is_dir() and p.name.isdigit()], key=lambda p: int(p.name)):
    path = n_dir / "history_effect_bayes_factors.csv"
    if path.exists():
        df = pd.read_csv(path)
        df["n_cell"] = int(n_dir.name)
        rows.append(df)

beta_sd = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
if beta_sd.empty:
    print("No beta Savage-Dickey files found yet.")
else:
    beta_sd["log10_BF10"] = np.log10(beta_sd["BF10"].replace({np.inf: np.nan}))
    display(beta_sd.sort_values(["parameter", "n_cell"]))

In [ ]:
if not beta_sd.empty:
    fig, ax = plt.subplots(figsize=(6.2, 4.0), dpi=220)

    for param, sub in beta_sd.sort_values("n_cell").groupby("parameter"):
        y = np.log10(sub["BF10"].replace({np.inf: np.nan}))
        ax.plot(sub["n_cell"], y, marker="o", label=PARAM_LABELS.get(param, param))

    ax.axhline(0.0, color="0.4", linestyle="--", linewidth=1.0)
    ax.set_xscale("log")
    ax.set_xlabel("Number of in-silico NK cells")
    ax.set_ylabel(r"$\log_{10} BF_{10}$ for nonzero beta")
    ax.grid(alpha=0.25)
    ax.legend(frameon=False)
    fig.tight_layout()

    out_path = figure_dir / "storyboard_beta_savage_dickey_trajectory.svg"
    fig.savefig(out_path, bbox_inches="tight", transparent=True)
    display(out_path)

## Next Code Files To Write

After this storyboard feels right, the next implementation should mirror section 1's Bayes-factor trajectory script:

1. `section_3/script/operation_2_bf_trajectory.py`
   - scenario grid above
   - cumulative population per scenario/replicate
   - fit all four `ModelSpec`s per sample size
   - save posterior, summary, log evidence, and beta Savage-Dickey where available

2. `section_3/script/run_2_bf_trajectory.sh`
   - clean user settings block
   - sample sizes, replicates, SMC particles, chains, `n_quad`

3. `section_3/notebooks/plot_2_bf_trajectory.ipynb`
   - eta heterogeneity log10 BF trajectory
   - beta Savage-Dickey log10 BF trajectory
   - confusion/recovery table across scenarios and sample sizes